# 04_feature_engineering

3주차 B팀: `monthly_merged.csv`를 기준으로 위험 지수와 모델링용 데이터셋을 생성합니다.

입력 파일: `data/processed/monthly_merged.csv`  
출력 파일: `data/processed/modeling_dataset.csv`


In [1]:


from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()


if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = PROCESSED_DIR / "monthly_merged.csv"
OUTPUT_PATH = PROCESSED_DIR / "modeling_dataset.csv"

print("프로젝트 루트:", ROOT)
print("입력 파일:", INPUT_PATH)
print("출력 파일:", OUTPUT_PATH)


df = pd.read_csv(INPUT_PATH, encoding="utf-8-sig")


df["contract_month"] = pd.to_datetime(df["contract_month"])
df = df.sort_values(["gu", "contract_month"]).reset_index(drop=True)

print("monthly_merged shape:", df.shape)

# 가격 차이
df["price_gap"] = df["avg_sale_price"] - df["avg_jeonse_deposit"]
df["purchase_burden_gap"] = df["price_gap"]

# 괴리 방향
df["gap_direction"] = np.where(
    df["price_gap"] > 0,
    "매매가>전세가",
    np.where(df["price_gap"] < 0, "전세가>매매가", "같음")
)

# 방향별 괴리율
df["sale_over_jeonse_gap_ratio"] = (
    np.maximum(df["avg_sale_price"] - df["avg_jeonse_deposit"], 0)
    / df["avg_sale_price"].replace(0, np.nan)
)

df["jeonse_over_sale_gap_ratio"] = (
    np.maximum(df["avg_jeonse_deposit"] - df["avg_sale_price"], 0)
    / df["avg_jeonse_deposit"].replace(0, np.nan)
)

df["gap_abs_ratio"] = (
    (df["avg_sale_price"] - df["avg_jeonse_deposit"]).abs()
    / np.maximum(df["avg_sale_price"], df["avg_jeonse_deposit"]).replace(0, np.nan)
)

# 월세화 변화량
df["monthly_ratio_diff"] = df.groupby("gu")["monthly_ratio"].diff()
df["monthly_ratio_pct_change"] = df.groupby("gu")["monthly_ratio"].pct_change()
df["monthly_shift_abs"] = df["monthly_ratio_diff"].abs()

# 매매가 상승 가속도
df["sale_growth_acceleration"] = df.groupby("gu")["sale_growth_1m"].diff()

# 같은 월 서울 평균 대비 상승률
market_growth = df.groupby("contract_month")["sale_growth_1m"].transform("mean")
df["sale_growth_vs_market"] = df["sale_growth_1m"] - market_growth


required_cols = [
    "jeonse_rate",
    "gap_rate",
    "sale_growth_1m",
    "jeonse_growth_1m",
    "growth_gap_1m",
    "sale_volume_growth_1m",
    "rent_volume_growth_1m",
    "monthly_ratio",
    "monthly_ratio_diff",
]

model_df = df.dropna(subset=required_cols).copy()

print("모델링 데이터 shape:", model_df.shape)
print("결측치 확인:")
print(model_df[required_cols].isnull().sum())


def minmax_score(series):
    """
    0~1 범위로 min-max 정규화한다.
    모든 값이 같은 경우 0으로 처리한다.
    """
    s = pd.to_numeric(series, errors="coerce")
    min_value = s.min()
    max_value = s.max()

    if pd.isna(min_value) or pd.isna(max_value) or max_value == min_value:
        return pd.Series(0, index=s.index)

    return (s - min_value) / (max_value - min_value)


model_df["low_jeonse_rate_score"] = 1 - minmax_score(model_df["jeonse_rate"])

# 값이 높을수록 위험하거나 주의가 필요한 변수
model_df["gap_rate_score"] = minmax_score(model_df["gap_rate"])
model_df["sale_growth_score"] = minmax_score(model_df["sale_growth_1m"])
model_df["growth_gap_score"] = minmax_score(model_df["growth_gap_1m"])
model_df["sale_volume_change_score"] = minmax_score(model_df["sale_volume_growth_1m"].abs())
model_df["rent_volume_change_score"] = minmax_score(model_df["rent_volume_growth_1m"].abs())
model_df["monthly_ratio_score"] = minmax_score(model_df["monthly_ratio"])
model_df["monthly_shift_score"] = minmax_score(model_df["monthly_ratio_diff"].abs())

# 전세가 상승률은 낮거나 정체될수록 실수요가 매매가를 덜 받쳐주는 것으로 해석
model_df["jeonse_weakness_score"] = 1 - minmax_score(model_df["jeonse_growth_1m"])

model_df["resident_risk_index"] = (
    0.25 * model_df["low_jeonse_rate_score"]
    + 0.25 * model_df["gap_rate_score"]
    + 0.15 * model_df["sale_growth_score"]
    + 0.15 * model_df["growth_gap_score"]
    + 0.10 * model_df["monthly_ratio_score"]
    + 0.10 * model_df["monthly_shift_score"]
)

model_df["investor_risk_index"] = (
    0.20 * model_df["gap_rate_score"]
    + 0.15 * model_df["low_jeonse_rate_score"]
    + 0.20 * model_df["sale_growth_score"]
    + 0.20 * model_df["growth_gap_score"]
    + 0.15 * model_df["sale_volume_change_score"]
    + 0.10 * model_df["monthly_shift_score"]
)


gap_q75 = model_df["gap_rate"].quantile(0.75)
monthly_shift_q75 = model_df["monthly_ratio_diff"].abs().quantile(0.75)

model_df["sale_stagnation_flag"] = (model_df["sale_growth_1m"].abs() <= 0.01).astype(int)
model_df["jeonse_stagnation_flag"] = (model_df["jeonse_growth_1m"].abs() <= 0.01).astype(int)
model_df["volume_drop_flag"] = (model_df["sale_volume_growth_1m"] < 0).astype(int)
model_df["high_gap_flag"] = (model_df["gap_rate"] >= gap_q75).astype(int)
model_df["rent_shift_flag"] = (model_df["monthly_ratio_diff"].abs() >= monthly_shift_q75).astype(int)

model_df["stagnation_score"] = (
    0.25 * model_df["sale_stagnation_flag"]
    + 0.20 * model_df["jeonse_stagnation_flag"]
    + 0.25 * model_df["volume_drop_flag"]
    + 0.20 * model_df["high_gap_flag"]
    + 0.10 * model_df["rent_shift_flag"]
)

def make_stagnation_level(score):
    if score <= 0.20:
        return "활발 관찰 구간"
    elif score <= 0.40:
        return "완만 정체 구간"
    elif score <= 0.60:
        return "안정 정체 구간"
    elif score <= 0.80:
        return "신중 정체 구간"
    else:
        return "거래 위축 정체 구간"

model_df["stagnation_level"] = model_df["stagnation_score"].apply(make_stagnation_level)


def quantile_grade(score, labels):
    """
    분위수 기준 5등급 생성
    labels는 낮은 위험 → 높은 위험 순서로 입력한다.
    """
    q20 = score.quantile(0.2)
    q40 = score.quantile(0.4)
    q60 = score.quantile(0.6)
    q80 = score.quantile(0.8)

    def grade(x):
        if x <= q20:
            return labels[0]
        elif x <= q40:
            return labels[1]
        elif x <= q60:
            return labels[2]
        elif x <= q80:
            return labels[3]
        else:
            return labels[4]

    return score.apply(grade)

model_df["resident_risk_grade"] = quantile_grade(
    model_df["resident_risk_index"],
    ["거주 안정권", "매수 검토 가능", "관찰 필요", "신중 매수 구간", "우선 점검 구간"]
)

model_df["investor_risk_grade"] = quantile_grade(
    model_df["investor_risk_index"],
    ["안정 관찰", "선별 검토 가능", "조건부 관찰", "신중 투자 구간", "우선 점검 구간"]
)

# 종합 위험 점수
model_df["total_risk_score"] = (
    model_df["resident_risk_index"] + model_df["investor_risk_index"]
) / 2

model_df["risk_grade"] = quantile_grade(
    model_df["total_risk_score"],
    ["낮음", "안정", "보통", "주의", "높음"]
)

# 상위 10% 경고 라벨
warning_cut = model_df["total_risk_score"].quantile(0.9)
model_df["warning_flag"] = (model_df["total_risk_score"] >= warning_cut).astype(int)


risk_cut = model_df["total_risk_score"].quantile(0.75)
model_df["risk_target"] = (model_df["total_risk_score"] >= risk_cut).astype(int)


model_df["contract_month"] = model_df["contract_month"].dt.strftime("%Y-%m")


model_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("저장 완료:", OUTPUT_PATH)
print("최종 데이터 shape:", model_df.shape)

print("\n위험 등급 분포:")
print(model_df["risk_grade"].value_counts())

print("\n위험 타깃 분포:")
print(model_df["risk_target"].value_counts())

print("\n정체도 단계 분포:")
print(model_df["stagnation_level"].value_counts())


프로젝트 루트: /Users/min/RealEstate-Risk-Radar
입력 파일: /Users/min/RealEstate-Risk-Radar/data/processed/monthly_merged.csv
출력 파일: /Users/min/RealEstate-Risk-Radar/data/processed/modeling_dataset.csv
monthly_merged shape: (900, 23)
모델링 데이터 shape: (875, 34)
결측치 확인:
jeonse_rate              0
gap_rate                 0
sale_growth_1m           0
jeonse_growth_1m         0
growth_gap_1m            0
sale_volume_growth_1m    0
rent_volume_growth_1m    0
monthly_ratio            0
monthly_ratio_diff       0
dtype: int64
저장 완료: /Users/min/RealEstate-Risk-Radar/data/processed/modeling_dataset.csv
최종 데이터 shape: (875, 58)

위험 등급 분포:
risk_grade
높음    175
주의    175
보통    175
안정    175
낮음    175
Name: count, dtype: int64

위험 타깃 분포:
risk_target
0    656
1    219
Name: count, dtype: int64

정체도 단계 분포:
stagnation_level
활발 관찰 구간       353
완만 정체 구간       304
안정 정체 구간       177
신중 정체 구간        38
거래 위축 정체 구간      3
Name: count, dtype: int64
